# Mission 3 — 2-2 실험: 오심 양성 가중치만 1.25배

이번 실험은 KLUE-RoBERTa Base, 발화 경계 SEP + First-512, label-attention head,
Weighted BCE, 4 epoch, **threshold 0.5 고정**을 유지합니다. 바꾸는 요인은 **오심의
양성 손실 가중치 하나**입니다. 두통·전신쇠약을 포함한 나머지 여덟 증상의 가중치는
2-1 W0와 동일합니다. TF-IDF 앙상블이나 threshold 튜닝은 하지 않습니다.

기존 W0는 백업 예측을 재사용하고 새 조건만 학습합니다. 맨 아래의 `RUN_SEED`를
42, 123, 2026 중 하나로 지정해 **한 번에 seed 하나만** 실행하세요. 세 seed는
사전에 고정한 반복 실험입니다. 한 seed가 끝나면 6번 비교 셀과 7번 다운로드 셀을
즉시 실행해 작은 결과 TAR를 보관하세요. 세 결과를 모두 보관해 비교합니다.

첫 실행 때 Colab 파일 패널에서 아래 **세 파일을 `/content`에 업로드**하세요.

1. `mission3_train_json.zip` (Mac: `Data AI Creater Camp/data/`)
2. `m3_targeted_weight_results.tar.gz` (Mac: `Downloads/`, seed 42 W0·분할)
3. `m3_2_1_results.tar` (Mac: `Downloads/`, seed 123·2026 W0)

학교 Colab GPU를 사용할 수 있는 **Drive 미사용** 버전입니다. 체크포인트는 `/content`에
매 epoch 저장되어 같은 VM에서 셀 재실행 시 이어갈 수 있습니다. VM 자체가 종료되면
체크포인트가 사라지고 해당 seed는 처음부터 다시 학습해야 합니다. 맥을 켜 두어도
Colab VM 종료를 막을 수 없습니다. 패키지 설치 셀을 실행했다면 런타임을 재시작한
뒤 설치 셀은 건너뛰고 나머지를 실행하세요.

이번 조건은 이미 확인한 2-1 내부 검증 결과를 바탕으로 고른 **후속 탐색**입니다.
내부 검증 점수가 높아도 독립적인 최종 성능으로 해석하지 마세요.


In [ ]:
%pip install -q "transformers==5.16.1" "accelerate>=1,<2" "scikit-learn>=1.5,<2" "pandas>=2.2,<3" safetensors "joblib>=1.4,<2"


## 1. 환경·입력 파일 확인


In [ ]:
from pathlib import Path
import json, zipfile, tarfile, hashlib, gc, inspect, platform, io
import importlib.metadata as im
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset
from sklearn.metrics import f1_score, precision_recall_fscore_support
from transformers import (AutoTokenizer, RobertaModel, RobertaPreTrainedModel,
                          TrainingArguments, Trainer, set_seed, RobertaConfig)
from transformers.modeling_outputs import SequenceClassifierOutput
from google.colab import files

# 한 번에 한 seed만 실행하고 다운로드합니다. 다음 실행 때 123, 2026으로 바꾸세요.
RUN_SEED = 42
assert RUN_SEED in [42, 123, 2026]
SEEDS_TO_RUN = [RUN_SEED]
TRAIN_ZIP = Path("/content/mission3_train_json.zip")
OLD_BACKUP = Path("/content/m3_targeted_weight_results.tar.gz")
PRIOR_MULTI = Path("/content/m3_2_1_results.tar")
OUT = Path("/content/m3_2_2_nausea_only")
OUT.mkdir(parents=True, exist_ok=True)
MANIFEST = OUT / "split_manifest.csv"
TARGETS = ["고열","구토","두통","복통","어지러움","열상","오심","전신쇠약","호흡곤란"]
MODEL_NAME = "klue/roberta-base"
assert torch.cuda.is_available(), "먼저 GPU 런타임을 선택하세요."
for path in [TRAIN_ZIP, OLD_BACKUP, PRIOR_MULTI]:
    assert path.is_file(), f"Colab 파일 패널에서 /content에 업로드하세요: {path}"
ENV = {"python": platform.python_version(), **{n: im.version(n) for n in
       ["torch","transformers","accelerate","numpy","pandas","scikit-learn"]}}
print("GPU:", torch.cuda.get_device_name(0), "환경:", ENV)
print("이번 seed:", RUN_SEED, "임시 체크포인트 경로:", OUT)
with tarfile.open(OLD_BACKUP, "r:gz") as archive:
    manifest_bytes = archive.extractfile("split_manifest.csv").read()
    reference_config = json.load(archive.extractfile("w0_current_weight/run_config.json"))
if MANIFEST.exists():
    assert MANIFEST.read_bytes() == manifest_bytes, "분할이 변경됐습니다."
else:
    MANIFEST.write_bytes(manifest_bytes)
REVISION = reference_config["model_revision"]
assert REVISION and reference_config["training"]["seed"] == 42

def reference_w0(seed):
    if seed == 42:
        path, mode, prefix = OLD_BACKUP, "r:gz", "w0_current_weight"
    else:
        path, mode, prefix = PRIOR_MULTI, "r", f"seed_{seed}/w0_current_weight"
    with tarfile.open(path, mode) as archive:
        config = json.load(archive.extractfile(f"{prefix}/run_config.json"))
        raw = archive.extractfile(f"{prefix}/dev_predictions.npz").read()
    with np.load(io.BytesIO(raw), allow_pickle=False) as z:
        predictions = {k: z[k].copy() for k in z.files}
    assert config["training"]["seed"] == seed
    assert config["manifest_sha256"] == hashlib.sha256(manifest_bytes).hexdigest()
    assert list(predictions["targets"]) == TARGETS
    return config, predictions

w0_config, w0_predictions = reference_w0(RUN_SEED)
print("기존 W0 예측과 설정 읽기 완료")


## 2. 동일 분할과 통화 텍스트 복원


In [ ]:
rows = []
with zipfile.ZipFile(TRAIN_ZIP) as z:
    for name in sorted(z.namelist()):
        p = Path(name)
        if "__MACOSX" in p.parts or p.name.startswith("._") or p.suffix.lower() != ".json":
            continue
        d = json.loads(z.read(name))
        texts = [u.get("text", "") for u in d["utterances"]]
        assert all(isinstance(t, str) for t in texts)
        assert isinstance(d["symptom"], list)
        rows.append({"file_name": p.name, "text": " ".join(texts), "utterance_texts": texts,
                     **{t: int(t in d["symptom"]) for t in TARGETS}})
df = pd.DataFrame(rows)
split = pd.read_csv(MANIFEST)
assert len(df) == 29200 and not df.file_name.duplicated().any()
assert not split.file_name.duplicated().any()
assert set(split.file_name) == set(df.file_name)
assert set(split.split) == {"train","dev_valid"}
ordered = split.merge(df, on="file_name", how="left", validate="one_to_one", sort=False)
train = ordered[ordered.split == "train"].reset_index(drop=True)
valid = ordered[ordered.split == "dev_valid"].reset_index(drop=True)
assert (len(train), len(valid)) == (23408, 5792)
assert hashlib.sha256(MANIFEST.read_bytes()).hexdigest() == reference_config["manifest_sha256"]
print("내부 학습:", train.shape, "내부 검증:", valid.shape)
print("manifest SHA256:", hashlib.sha256(MANIFEST.read_bytes()).hexdigest())

# seed=42 결과의 정답·행 순서도 현재 검증셋과 일치해야 합니다.
for experiment in ["w0_current_weight", "w1_targeted_low_recall_x125"]:
    with tarfile.open(OLD_BACKUP, "r:gz") as a:
        b = a.extractfile(f"{experiment}/dev_predictions.npz").read()
    with np.load(io.BytesIO(b), allow_pickle=False) as z:
        assert list(z["targets"]) == TARGETS
        assert np.array_equal(z["file_names"], valid.file_name.to_numpy())
        assert np.array_equal(z["y_true"], valid[TARGETS].to_numpy())
print("기존 seed=42와 동일한 데이터·검증 순서 확인")

assert np.array_equal(w0_predictions["file_names"], valid.file_name.to_numpy())
assert np.array_equal(w0_predictions["y_true"], valid[TARGETS].to_numpy())
assert w0_predictions["probs"].shape == (5792, 9)
assert np.isfinite(w0_predictions["probs"]).all()
print("이번 seed의 W0와 검증 순서·정답 일치 확인")


## 3. 발화 경계 First-512와 오심 단독 가중치


In [ ]:
import re, unicodedata
MODES = ["raw", "boundary", "boundary_normalized"]
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=REVISION)
def normalize_utterance(text):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", text)).strip()

def encode_call(utterances, mode):
    assert mode in MODES
    if mode == "raw":
        body = tokenizer(" ".join(utterances), add_special_tokens=False, truncation=False)["input_ids"]
    else:
        body = []
        for i, text in enumerate(utterances):
            if i:
                body.append(tokenizer.sep_token_id)
            if mode == "boundary_normalized":
                text = normalize_utterance(text)
            body.extend(tokenizer(text, add_special_tokens=False, truncation=False)["input_ids"])
    full_length = len(body) + 2
    return [tokenizer.cls_token_id] + body[:510] + [tokenizer.sep_token_id], full_length

class Calls(Dataset):
    def __init__(self, frame, mode):
        encoded = [encode_call(u, mode) for u in frame.utterance_texts]
        self.ids = [x[0] for x in encoded]
        self.lengths = np.array([x[1] for x in encoded])
        self.labels = frame[TARGETS].to_numpy(dtype=np.float32)
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        return {"input_ids": self.ids[i], "labels": self.labels[i]}

def collate(batch):
    x = tokenizer.pad([{"input_ids": b["input_ids"]} for b in batch], return_tensors="pt")
    x["labels"] = torch.tensor(np.stack([b["labels"] for b in batch]), dtype=torch.float32)
    return x

assert normalize_utterance("토할  것 같아요.\n토는 안 했어요.") == "토할 것 같아요. 토는 안 했어요."
assert normalize_utterance("오심 구토 아니요 없어요") == "오심 구토 아니요 없어요"
for utterances in train.utterance_texts.iloc[:30]:
    actual, _ = encode_call(utterances, "raw")
    expected = tokenizer(" ".join(utterances), truncation=True, max_length=512)["input_ids"]
    assert actual == expected, "P0가 이전 입력과 다릅니다."
for mode in MODES:
    ids_test, _ = encode_call(["", "토는 안 했어요.", "속이 울렁거려요."], mode)
    assert len(ids_test) <= 512 and ids_test[0] == tokenizer.cls_token_id

# 이 단계부터 사용하는 전처리는 boundary 하나뿐입니다.
train_ds, valid_ds = Calls(train, "boundary"), Calls(valid, "boundary")
y_train = train[TARGETS].to_numpy(dtype=np.float32)
positive = y_train.sum(axis=0)
negative = len(train) - positive
assert (positive > 0).all(), "양성 0개 클래스가 있습니다."
BASE_WEIGHTS = np.clip(np.sqrt(negative / positive), 1., 3.).astype(np.float32)
TARGETED_SYMPTOMS = ["오심"]
BOOST_FACTOR = 1.25
BOOST_CAP = 4.0
BOOSTED_WEIGHTS = BASE_WEIGHTS.copy()
for symptom in TARGETED_SYMPTOMS:
    i = TARGETS.index(symptom)
    BOOSTED_WEIGHTS[i] = min(BOOSTED_WEIGHTS[i] * BOOST_FACTOR, BOOST_CAP)

EXPERIMENTS = ["nausea_only_x125"]
WEIGHTS_BY_EXPERIMENT = {"nausea_only_x125": BOOSTED_WEIGHTS}
weight_table = pd.DataFrame({"symptom": TARGETS, "positive": positive, "negative": negative,
    "w0_current_weight": BASE_WEIGHTS, "nausea_only_weight": BOOSTED_WEIGHTS,
    "is_targeted": [t in TARGETED_SYMPTOMS for t in TARGETS]})
display(weight_table)
weight_table.to_csv(OUT / "training_pos_weights.csv", index=False)
(OUT / "weight_rule.json").write_text(json.dumps({"base_rule": "clip(sqrt(negative/positive),1,3)",
    "targeted_symptoms": TARGETED_SYMPTOMS, "boost_factor": BOOST_FACTOR, "boost_cap": BOOST_CAP,
    "source": "internal train only", "w0_weights": BASE_WEIGHTS.tolist(),
    "nausea_only_weights": BOOSTED_WEIGHTS.tolist()}, ensure_ascii=False, indent=2))
np.testing.assert_allclose(BASE_WEIGHTS, reference_config["pos_weight"], rtol=0, atol=1e-7)
print("동일 boundary First-512 입력 준비:", len(train_ds), len(valid_ds))

# 고정 임계값 평가 함수: 학습·요약 셀보다 먼저 정의해야 합니다.
def label_report(y, probs):
    pred = probs >= .5
    p, r, f, support = precision_recall_fscore_support(
        y, pred, average=None, zero_division=0)
    return pd.DataFrame({"symptom": TARGETS, "precision": p, "recall": r,
                         "f1": f, "support": support,
                         "predicted_positive": pred.sum(axis=0)})

for symptom in TARGETS:
    i = TARGETS.index(symptom)
    if symptom != "오심":
        assert BOOSTED_WEIGHTS[i] == BASE_WEIGHTS[i]
print("오심만 변경:", BASE_WEIGHTS[TARGETS.index("오심")],
      "->", BOOSTED_WEIGHTS[TARGETS.index("오심")])


## 4. 이전 실험과 동일한 모델·손실 함수 검증


In [ ]:
class SymptomModel(RobertaPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.roberta = RobertaModel(config, add_pooling_layer=False)
        h = config.hidden_size
        self.head_type = getattr(config, "head_type", "cls")
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.dense = nn.Linear(h, h)
        self.out_proj = nn.Linear(h, config.num_labels)
        if self.head_type == "label_attention":
            self.attention = nn.Linear(h, config.num_labels, bias=False)
        # Keep loss weights as ordinary config values, not a tensor buffer subject
        # to from_pretrained's meta-device / missing-weight initialization.
        values = getattr(config, "loss_pos_weight", [1.] * config.num_labels)
        if len(values) != config.num_labels or any(not np.isfinite(v) or v <= 0 for v in values):
            raise ValueError("Invalid loss weights")
        config.loss_pos_weight = [float(v) for v in values]
        self.post_init()
    def forward(self, input_ids=None, attention_mask=None, labels=None):
        hidden = self.roberta(input_ids=input_ids, attention_mask=attention_mask,
                              return_dict=True).last_hidden_state
        mask = attention_mask.bool().clone()
        mask[:,0] = False
        last = attention_mask.sum(1).long() - 1
        mask[torch.arange(len(mask), device=mask.device), last] = False
        empty = ~mask.any(dim=1)
        mask[empty,0] = True
        if self.head_type == "cls":
            pooled = hidden[:,0]
        elif self.head_type == "mean":
            pooled = (hidden * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True)
        else:
            scores = self.attention(hidden).float().masked_fill(~mask.unsqueeze(-1), -1e9)
            weights = scores.softmax(dim=1).to(hidden.dtype)
            pooled = torch.einsum("blc,blh->bch", weights, hidden)
        x = self.dropout(torch.tanh(self.dense(self.dropout(pooled))))
        if self.head_type == "label_attention":
            logits = (x * self.out_proj.weight.unsqueeze(0)).sum(-1) + self.out_proj.bias
        else:
            logits = self.out_proj(x)
        if not torch.isfinite(logits).all():
            raise FloatingPointError("Non-finite logits: stop before saving results")
        loss = None
        if labels is not None:
            if labels.shape != logits.shape or not ((labels == 0) | (labels == 1)).all():
                raise ValueError("Labels must be binary and match logits")
            # Construct on the actual device AFTER loading, on each forward pass.
            pos_weight = torch.tensor(self.config.loss_pos_weight, device=logits.device,
                                      dtype=torch.float32)
            loss = nn.functional.binary_cross_entropy_with_logits(
                logits.float(), labels.float(), pos_weight=pos_weight)
            if not torch.isfinite(loss) or loss.detach().item() > 100:
                raise FloatingPointError("Abnormal BCE loss; inspect initialization and labels")
        return SequenceClassifierOutput(loss=loss, logits=logits)

def compute_metrics(p):
    logits = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
    return {"macro_f1": f1_score(p.label_ids, logits >= 0, average="macro", zero_division=0)}

# 다운로드·학습 없는 구조 검사
tiny = RobertaConfig(vocab_size=32, hidden_size=24, num_hidden_layers=1,
                     num_attention_heads=4, intermediate_size=48, num_labels=9)
for kind in ["cls","mean","label_attention"]:
    tiny.head_type = kind
    m = SymptomModel(tiny).eval()
    with torch.no_grad():
        v = m(torch.tensor([[0,5,6,2,1],[0,2,1,1,1]]),
              torch.tensor([[1,1,1,1,0],[1,1,0,0,0]]), torch.zeros(2,9))
    assert v.logits.shape == (2,9) and torch.isfinite(v.loss)
del m
print("head 구조 검사 OK")

# Real save/load regression test using a tiny local model (no download).
import tempfile
def verify_loss(model, batch, expected_weights):
    model.eval()
    with torch.no_grad():
        result = model(**batch)
        z, y = result.logits.float(), batch["labels"].float()
        w = torch.tensor(expected_weights, dtype=torch.float32, device=z.device)
        # Independent BCE identity, with no reference to model.config weights.
        manual = ((1-y)*nn.functional.softplus(z) + w*y*nn.functional.softplus(-z)).mean()
    torch.testing.assert_close(result.loss, manual, rtol=1e-5, atol=1e-6)
    assert model.config.loss_pos_weight == list(expected_weights)
    return result

for test_weights in [[1.]*9, [2.]*9]:
    tiny.head_type = "label_attention"
    tiny.loss_pos_weight = test_weights
    test_model = SymptomModel(tiny).eval()
    test_batch = {"input_ids": torch.tensor([[0,5,6,2],[0,7,8,2]]),
                  "attention_mask": torch.ones(2,4,dtype=torch.long),
                  "labels": torch.tensor([[1,0,1,0,1,0,1,0,1],[0,1,0,1,0,1,0,1,0]],dtype=torch.float32)}
    before = verify_loss(test_model, test_batch, test_weights)
    with tempfile.TemporaryDirectory() as test_dir:
        test_model.save_pretrained(test_dir)
        loaded = SymptomModel.from_pretrained(test_dir).eval()
        after = verify_loss(loaded, test_batch, test_weights)
        torch.testing.assert_close(before.logits, after.logits)
    del test_model, loaded
print("BCE independent calculation + save/load checks passed")

from transformers import TrainerCallback
class FiniteGradientCheck(TrainerCallback):
    def on_pre_optimizer_step(self, args, state, control, model=None, **kwargs):
        for name, parameter in model.named_parameters():
            if parameter.grad is not None and not torch.isfinite(parameter.grad).all():
                raise FloatingPointError(f"Non-finite gradient: {name}")
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics and metrics.get("eval_macro_f1") == 0:
            raise RuntimeError("Epoch evaluation F1 is zero. Stop and inspect; do not run L1 yet.")


## 5. 지정 seed의 오심 단독 가중치 모델 학습

학습 종료 후 결과를 바로 내려받습니다. 같은 VM에서만 epoch 체크포인트 재개가 가능합니다.


In [ ]:
from transformers import TrainerCallback

class CheckpointReady(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        checkpoint = Path(args.output_dir) / f"checkpoint-{state.global_step}"
        assert checkpoint.is_dir(), f"체크포인트 폴더 없음: {checkpoint}"
        (checkpoint / ".checkpoint_complete").write_text(str(state.global_step))

def completed_run(seed, experiment):
    path = OUT / f"seed_{seed}" / experiment
    required = [path / "metrics.json", path / "dev_predictions.npz", path / "run_config.json"]
    if not all(p.is_file() for p in required):
        return False
    with np.load(path / "dev_predictions.npz", allow_pickle=False) as z:
        assert list(z["targets"]) == TARGETS
        assert np.array_equal(z["file_names"], valid.file_name.to_numpy())
        assert np.array_equal(z["y_true"], valid[TARGETS].to_numpy())
        assert np.isfinite(z["probs"]).all()
        score = float(f1_score(z["y_true"], z["probs"] >= .5, average="macro", zero_division=0))
    metrics = json.loads((path / "metrics.json").read_text())
    assert abs(score - metrics["fixed_threshold_macro_f1"]) < 1e-9
    return True

def usable_checkpoints(folder):
    needed = {"trainer_state.json", "optimizer.pt", "scheduler.pt", "rng_state.pth",
              ".checkpoint_complete"}
    for p in sorted(folder.glob("checkpoint-*"),
                    key=lambda x: int(x.name.split("-")[-1]), reverse=True):
        if needed.issubset({x.name for x in p.iterdir()}) and any(
            (p / n).is_file() and (p / n).stat().st_size > 100_000_000
            for n in ["model.safetensors", "pytorch_model.bin"]):
            yield p

def finish_saved_prediction(seed, experiment, out):
    """Finish a run that crashed after final predictions were saved."""
    prediction_path = out / "dev_predictions.npz"
    if not prediction_path.is_file():
        return False
    with np.load(prediction_path, allow_pickle=False) as z:
        assert list(z["targets"]) == TARGETS
        assert np.array_equal(z["file_names"], valid.file_name.to_numpy())
        assert np.array_equal(z["y_true"], valid[TARGETS].to_numpy())
        y, probs = z["y_true"].copy(), z["probs"].copy()
    assert probs.shape == y.shape and np.isfinite(probs).all()
    label_report(y, probs).to_csv(out / "per_label.csv", index=False)
    checkpoints = list(usable_checkpoints(out / "checkpoints"))
    if checkpoints:
        state = json.loads((checkpoints[0] / "trainer_state.json").read_text())
        (out / "training_log.json").write_text(json.dumps(state.get("log_history", []), indent=2))
    result = {"fixed_threshold": .5,
              "fixed_threshold_macro_f1": float(f1_score(y, probs >= .5, average="macro", zero_division=0)),
              "seed": seed, "experiment": experiment, "evaluation": "internal validation",
              "selection": "final_epoch"}
    temp_metrics = out / "metrics.tmp.json"
    temp_metrics.write_text(json.dumps(result, ensure_ascii=False, indent=2))
    temp_metrics.replace(out / "metrics.json")
    print("저장된 최종 예측으로 결과 복구:", result)
    return True

def run_experiment(seed, experiment):
    assert seed in SEEDS_TO_RUN and experiment in EXPERIMENTS
    out = OUT / f"seed_{seed}" / experiment
    out.mkdir(parents=True, exist_ok=True)
    if completed_run(seed, experiment):
        print("완료 결과 재사용:", out)
        return
    if finish_saved_prediction(seed, experiment, out):
        return
    checkpoints = list(usable_checkpoints(out / "checkpoints"))
    resume = str(checkpoints[0]) if checkpoints else None
    print("시작:", seed, experiment, "재개 위치:", resume or "처음부터")
    set_seed(seed)
    config = RobertaConfig.from_pretrained(MODEL_NAME, revision=REVISION)
    config.num_labels = len(TARGETS)
    config.id2label = dict(enumerate(TARGETS))
    config.label2id = {t:i for i,t in enumerate(TARGETS)}
    config.head_type = "label_attention"
    config.loss_pos_weight = WEIGHTS_BY_EXPERIMENT[experiment].tolist()
    model = SymptomModel.from_pretrained(MODEL_NAME, config=config, revision=REVISION)
    checks = sorted(set(int(np.flatnonzero(train[TARGETS].to_numpy()[:, j])[0]) for j in range(9)))
    batch = collate([train_ds[i] for i in checks])
    assert (batch["labels"].sum(0) > 0).all()
    verify_loss(model, batch, config.loss_pos_weight)
    model.train()
    digest = hashlib.sha256()
    for name, parameter in model.named_parameters():
        digest.update(name.encode())
        digest.update(parameter.detach().cpu().contiguous().numpy().tobytes())
    initial_hash = digest.hexdigest()
    assert initial_hash == w0_config["initial_parameter_sha256"], (
        "기존 W0와 초기 모델이 다릅니다. 모델 revision·seed·환경을 확인하세요.")
    paired_hash_path = OUT / f"seed_{seed}" / "initial_parameter_sha256.txt"
    if paired_hash_path.exists():
        assert paired_hash_path.read_text() == initial_hash
    else:
        paired_hash_path.write_text(initial_hash)
    kwargs = dict(output_dir=str(out / "checkpoints"), num_train_epochs=4,
                  per_device_train_batch_size=8, per_device_eval_batch_size=16,
                  gradient_accumulation_steps=1, learning_rate=2e-5, warmup_steps=1000,
                  weight_decay=.01, logging_steps=50, logging_first_step=True,
                  fp16=False, bf16=False, tf32=False, max_grad_norm=1.0,
                  report_to="none", seed=seed, data_seed=seed,
                  save_strategy="epoch", save_total_limit=1,
                  load_best_model_at_end=False)
    key = "eval_strategy" if "eval_strategy" in inspect.signature(TrainingArguments).parameters else "evaluation_strategy"
    kwargs[key] = "epoch"
    # 학습 핵심 조건이 기존 W0와 같은지 학습 전에 확인합니다.
    for setting in ["num_train_epochs", "per_device_train_batch_size",
                    "per_device_eval_batch_size", "learning_rate", "warmup_steps",
                    "weight_decay", "gradient_accumulation_steps", "seed", "data_seed"]:
        assert kwargs[setting] == w0_config["training"][setting], setting
    provenance = {"implementation": "loss_config_fp32_v2_nausea_only",
                  "head": "label_attention", "model": MODEL_NAME,
                  "model_revision": REVISION, "threshold": .5,
                  "loss": "weighted_bce", "seed": seed, "experiment": experiment,
                  "pos_weight": config.loss_pos_weight,
                  "initial_parameter_sha256": initial_hash,
                  "selection": "final_epoch", "input": "first_512",
                  "preprocessing": "boundary", "normalization": "none",
                  "targets": TARGETS, "manifest_sha256": hashlib.sha256(MANIFEST.read_bytes()).hexdigest(),
                  "training": kwargs}
    config_path = out / "run_config.json"
    if config_path.exists():
        assert json.loads(config_path.read_text()) == provenance, "저장된 실행 설정과 다릅니다. 중단하고 확인하세요."
    else:
        config_path.write_text(json.dumps(provenance, ensure_ascii=False, indent=2))
    trainer = Trainer(model=model, args=TrainingArguments(**kwargs), train_dataset=train_ds,
                      eval_dataset=valid_ds, data_collator=collate, compute_metrics=compute_metrics,
                      callbacks=[FiniteGradientCheck(), CheckpointReady()])
    trainer.train(resume_from_checkpoint=resume)
    logits = trainer.predict(valid_ds).predictions
    if isinstance(logits, tuple): logits = logits[0]
    probs = torch.sigmoid(torch.as_tensor(logits).float()).numpy()
    y = valid[TARGETS].to_numpy(dtype=int)
    assert probs.shape == y.shape and np.isfinite(probs).all()
    npz_temp = out / "dev_predictions.tmp.npz"
    np.savez_compressed(npz_temp, probs=probs, y_true=y,
                        file_names=valid.file_name.to_numpy(dtype=str), targets=np.asarray(TARGETS))
    npz_temp.replace(out / "dev_predictions.npz")
    label_report(y, probs).to_csv(out / "per_label.csv", index=False)
    (out / "training_log.json").write_text(json.dumps(trainer.state.log_history, indent=2))
    result = {"fixed_threshold": .5,
              "fixed_threshold_macro_f1": float(f1_score(y, probs >= .5, average="macro", zero_division=0)),
              "seed": seed, "experiment": experiment, "evaluation": "internal validation",
              "selection": "final_epoch"}
    temp_metrics = out / "metrics.tmp.json"
    temp_metrics.write_text(json.dumps(result, ensure_ascii=False, indent=2))
    temp_metrics.replace(out / "metrics.json")  # 완료 표시를 마지막에 기록
    print("완료:", result)
    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()

run_experiment(RUN_SEED, EXPERIMENTS[0])
assert completed_run(RUN_SEED, EXPERIMENTS[0])
print("학습·예측 저장 완료")


## 6. 기존 W0와 2-2 결과 비교

고정 threshold 0.5로만 평가합니다. 이 seed의 결과는 다른 seed 결과와 합쳐 판단하세요.


In [ ]:
experiment = EXPERIMENTS[0]
out = OUT / f"seed_{RUN_SEED}" / experiment
with np.load(out / "dev_predictions.npz", allow_pickle=False) as z:
    new = {k: z[k].copy() for k in z.files}
assert np.array_equal(w0_predictions["file_names"], new["file_names"])
assert np.array_equal(w0_predictions["y_true"], new["y_true"])
y = new["y_true"]
w0_score = float(f1_score(y, w0_predictions["probs"] >= .5, average="macro", zero_division=0))
new_score = float(f1_score(y, new["probs"] >= .5, average="macro", zero_division=0))
comparison = pd.DataFrame([{"seed": RUN_SEED, "w0_macro_f1": w0_score,
                            "nausea_only_macro_f1": new_score,
                            "nausea_only_minus_w0": new_score - w0_score}])
comparison.to_csv(OUT / f"seed_{RUN_SEED}" / "comparison.csv", index=False)
display(comparison)
w0_labels = label_report(y, w0_predictions["probs"]).set_index("symptom")
new_labels = label_report(y, new["probs"]).set_index("symptom")
label_comparison = pd.DataFrame({
    "symptom": TARGETS,
    "w0_precision": w0_labels.loc[TARGETS, "precision"].to_numpy(),
    "new_precision": new_labels.loc[TARGETS, "precision"].to_numpy(),
    "w0_recall": w0_labels.loc[TARGETS, "recall"].to_numpy(),
    "new_recall": new_labels.loc[TARGETS, "recall"].to_numpy(),
    "w0_f1": w0_labels.loc[TARGETS, "f1"].to_numpy(),
    "new_f1": new_labels.loc[TARGETS, "f1"].to_numpy(),
})
label_comparison.to_csv(OUT / f"seed_{RUN_SEED}" / "label_comparison.csv", index=False)
display(label_comparison)


## 7. 작은 결과 TAR 즉시 다운로드

실험이 끝나면 바로 실행하세요. 모델 체크포인트는 제외하고 예측·지표·설정·로그를 저장합니다.


In [ ]:
archive_path = Path(f"/content/m3_2_2_nausea_only_seed{RUN_SEED}.tar")
with tarfile.open(archive_path, "w") as archive:
    for path in sorted((OUT / f"seed_{RUN_SEED}").rglob("*")):
        if path.is_file() and "checkpoints" not in path.relative_to(OUT).parts:
            archive.add(path, arcname=str(path.relative_to(OUT)))
    for name in ["training_pos_weights.csv", "weight_rule.json"]:
        archive.add(OUT / name, arcname=name)
print("결과 묶음:", archive_path, "크기(MB):", round(archive_path.stat().st_size / 1e6, 2))
files.download(str(archive_path))


## 8. 선택: 이 seed의 모델 체크포인트 다운로드

재추론이나 학습 재개가 필요할 때만 실행하세요. 수 GB가 될 수 있고 브라우저 다운로드가 실패할 수 있습니다.


In [ ]:
DOWNLOAD_CHECKPOINT = False
if DOWNLOAD_CHECKPOINT:
    checkpoint_archive = Path(f"/content/m3_2_2_checkpoint_seed{RUN_SEED}.tar")
    with tarfile.open(checkpoint_archive, "w") as archive:
        for path in sorted((OUT / f"seed_{RUN_SEED}" / EXPERIMENTS[0] / "checkpoints").rglob("*")):
            if path.is_file():
                archive.add(path, arcname=str(path.relative_to(OUT)))
    print("체크포인트 크기(GB):", round(checkpoint_archive.stat().st_size / 1e9, 2))
    files.download(str(checkpoint_archive))
else:
    print("대형 체크포인트 다운로드 건너뜀")
